# Kubeflow: ML Pipelines on Kubernetes

## What Is Kubeflow?

Imagine your ML pipeline is a factory assembly line.  
Each station (data prep, training, evaluation, serving) is run by a worker.  
**Kubernetes** is the factory floor — it manages the workers (containers).  
**Kubeflow** is the factory manager for ML — it specializes Kubernetes for ML workloads.

**Kubeflow** is an open-source ML platform on Kubernetes that provides:
- **Kubeflow Pipelines (KFP)**: define ML workflows as Python code → runs on K8s
- **Katib**: automated hyperparameter tuning on K8s
- **KServe**: scalable model serving (formerly KFServing)
- **Notebooks**: JupyterHub managed by Kubernetes
- **Training operators**: distributed training for PyTorch, TensorFlow, MPI

## Resources

- **Docs**: [https://www.kubeflow.org/docs/](https://www.kubeflow.org/docs/)
- **KFP SDK**: [https://kubeflow-pipelines.readthedocs.io/](https://kubeflow-pipelines.readthedocs.io/)
- **YouTube — Kubeflow intro**: [https://www.youtube.com/watch?v=cHkfq9vZBmQ](https://www.youtube.com/watch?v=cHkfq9vZBmQ)
- **KFP tutorial**: [https://www.kubeflow.org/docs/components/pipelines/v2/tutorials/](https://www.kubeflow.org/docs/components/pipelines/v2/tutorials/)

## Installation

```bash
# KFP SDK — for defining and compiling pipelines
pip install kfp

# Full Kubeflow requires a Kubernetes cluster:
# - Google Kubernetes Engine (GKE): easiest
# - Amazon EKS
# - Local: kind or minikube
# Then install with: kubectl apply -k kubeflow/manifests/
```

**For this notebook**: We use `kfp` (Python SDK) to define and compile pipelines.  
The compiled YAML can be uploaded to any Kubeflow Pipelines UI.

In [ ]:
import json, os, tempfile
import numpy as np
import pandas as pd

try:
    import kfp
    from kfp import dsl
    from kfp.dsl import component, pipeline, Input, Output, Dataset, Model, Metrics
    KFP_AVAILABLE = True
    print(f"KFP version: {kfp.__version__}")
except ImportError:
    KFP_AVAILABLE = False
    print("kfp not installed — simulated output shown. Install: pip install kfp")

print("\nKubeflow Pipelines SDK is used ONLY for defining pipelines.")
print("Execution happens on a Kubernetes cluster (not needed for this notebook).")

## Core Concept 1: Components — The Building Blocks

A **component** is the basic unit of a Kubeflow pipeline.  
Each component:
- Runs inside its own **Docker container** (isolated environment)
- Takes **inputs** (parameters or artifacts like datasets)
- Produces **outputs** (artifacts like models or metrics)
- Is defined using the `@component` decorator

**Why containers?** Isolation — each step can have different Python environments, different GPUs, different resource requests.

In [ ]:
if KFP_AVAILABLE:
    # ── Define pipeline components ────────────────────────────────────────────

    @component(
        base_image="python:3.11",
        packages_to_install=["pandas", "scikit-learn", "numpy"]
    )
    def generate_data(
        n_samples: int,
        n_features: int,
        output_dataset: Output[Dataset],  # artifact: file written to this path
    ):
        """Generate synthetic classification data."""
        import pandas as pd
        import numpy as np
        from sklearn.datasets import make_classification

        X, y = make_classification(
            n_samples=n_samples, n_features=n_features,
            n_informative=n_features // 2, random_state=42
        )
        df = pd.DataFrame(X, columns=[f'f{i}' for i in range(n_features)])
        df['label'] = y
        df.to_csv(output_dataset.path, index=False)
        print(f"Generated {n_samples} samples, {n_features} features")


    @component(
        base_image="python:3.11",
        packages_to_install=["pandas", "scikit-learn", "numpy"]
    )
    def train_model(
        input_dataset: Input[Dataset],
        n_estimators: int,
        max_depth: int,
        output_model: Output[Model],
        output_metrics: Output[Metrics],
    ):
        """Train a RandomForest classifier."""
        import pandas as pd
        import pickle
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.model_selection import train_test_split
        from sklearn.metrics import accuracy_score, roc_auc_score

        df = pd.read_csv(input_dataset.path)
        X = df.drop('label', axis=1).values
        y = df['label'].values
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth)
        model.fit(X_train, y_train)

        acc = accuracy_score(y_test, model.predict(X_test))
        auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

        # Log metrics to Kubeflow UI
        output_metrics.log_metric('accuracy', acc)
        output_metrics.log_metric('roc_auc', auc)

        with open(output_model.path, 'wb') as f:
            pickle.dump(model, f)

        print(f"Model: acc={acc:.4f} auc={auc:.4f}")


    @component(
        base_image="python:3.11",
        packages_to_install=["pandas", "scikit-learn", "numpy"]
    )
    def evaluate_and_decide(
        model: Input[Model],
        metrics: Input[Metrics],
        min_auc_threshold: float,
    ) -> str:  # returns 'deploy' or 'reject'
        """Decide whether to deploy the model."""
        val_auc = metrics.metadata.get('roc_auc', 0)
        decision = 'deploy' if val_auc >= min_auc_threshold else 'reject'
        print(f"AUC={val_auc:.4f}, threshold={min_auc_threshold}: {decision.upper()}")
        return decision


    print("KFP components defined:")
    print("  - generate_data: outputs Dataset artifact")
    print("  - train_model: inputs Dataset, outputs Model + Metrics")
    print("  - evaluate_and_decide: inputs Model + Metrics, returns deploy/reject")

else:
    print("Component definition pattern (simulated):")
    print()
    print("  @component(")
    print("      base_image='python:3.11',")
    print("      packages_to_install=['pandas', 'scikit-learn']")
    print("  )")
    print("  def train_model(")
    print("      input_dataset: Input[Dataset],  # artifact passed in")
    print("      n_estimators: int,               # scalar parameter")
    print("      output_model: Output[Model],     # artifact written out")
    print("      output_metrics: Output[Metrics], # metrics logged to UI")
    print("  ):")
    print("      # All code runs INSIDE a Docker container on Kubernetes")
    print("      import sklearn  # available because packages_to_install")
    print("      model = RandomForestClassifier(n_estimators=n_estimators)")
    print("      ...")

## Core Concept 2: Pipelines — Connecting Components

In [ ]:
if KFP_AVAILABLE:
    @pipeline(
        name='ml-training-pipeline',
        description='End-to-end ML training with automatic deployment decision'
    )
    def ml_pipeline(
        n_samples:   int   = 2000,
        n_features:  int   = 20,
        n_estimators:int   = 100,
        max_depth:   int   = 8,
        min_auc:     float = 0.85,
    ):
        # Step 1: Generate data
        data_task = generate_data(
            n_samples=n_samples,
            n_features=n_features,
        )
        # Configure K8s resources for this component
        data_task.set_memory_limit('2Gi')
        data_task.set_cpu_limit('1')

        # Step 2: Train model (depends on Step 1's output)
        train_task = train_model(
            input_dataset=data_task.outputs['output_dataset'],
            n_estimators=n_estimators,
            max_depth=max_depth,
        )
        train_task.set_memory_limit('4Gi')
        train_task.set_cpu_limit('2')
        # Enable GPU: train_task.set_accelerator_type('NVIDIA_TESLA_T4').set_accelerator_limit(1)

        # Step 3: Evaluate
        eval_task = evaluate_and_decide(
            model=train_task.outputs['output_model'],
            metrics=train_task.outputs['output_metrics'],
            min_auc_threshold=min_auc,
        )

    # Compile pipeline to YAML
    output_path = os.path.join(tempfile.gettempdir(), 'ml_pipeline.yaml')
    kfp.compiler.Compiler().compile(ml_pipeline, output_path)
    print(f"Pipeline compiled to: {output_path}")

    # Show first 50 lines of compiled YAML
    with open(output_path) as f:
        lines = f.readlines()[:50]
    print("\nFirst 50 lines of compiled pipeline YAML:")
    print("".join(lines))

else:
    print("Pipeline definition (simulated):")
    print()
    print("  @pipeline(name='ml-training-pipeline')")
    print("  def ml_pipeline(n_samples=2000, n_estimators=100, min_auc=0.85):")
    print()
    print("      # Components are connected through their output/input artifacts")
    print("      data_task = generate_data(n_samples=n_samples)")
    print()
    print("      train_task = train_model(")
    print("          input_dataset=data_task.outputs['output_dataset'],")
    print("          n_estimators=n_estimators,")
    print("      )")
    print("      train_task.set_memory_limit('4Gi')  # K8s resource request")
    print("      train_task.set_cpu_limit('2')")
    print()
    print("  # Compile to YAML for upload to Kubeflow UI")
    print("  kfp.compiler.Compiler().compile(ml_pipeline, 'pipeline.yaml')")
    print()
    print("  # Or submit directly to a Kubeflow cluster")
    print("  client = kfp.Client(host='https://kubeflow.company.com')")
    print("  client.create_run_from_pipeline_func(ml_pipeline, arguments={...})")

## Core Concept 3: Katib — Hyperparameter Tuning on K8s

In [ ]:
# Katib Experiment configuration
katib_experiment = {
    "apiVersion": "kubeflow.org/v1beta1",
    "kind": "Experiment",
    "metadata": {
        "name": "random-forest-hpo",
        "namespace": "ml-team"
    },
    "spec": {
        "objective": {
            "type": "maximize",
            "goal": 0.95,
            "objectiveMetricName": "roc_auc",
            "additionalMetricNames": ["accuracy"]
        },
        "algorithm": {
            "algorithmName": "bayesianoptimization"
        },
        "parallelTrialCount": 4,   # run 4 trials simultaneously
        "maxTrialCount": 20,        # maximum 20 trials total
        "maxFailedTrialCount": 3,   # stop if 3 trials fail
        "parameters": [
            {
                "name": "n_estimators",
                "parameterType": "int",
                "feasibleSpace": {"min": "50", "max": "500", "step": "50"}
            },
            {
                "name": "max_depth",
                "parameterType": "int",
                "feasibleSpace": {"min": "3", "max": "20"}
            },
            {
                "name": "learning_rate",
                "parameterType": "double",
                "feasibleSpace": {"min": "0.001", "max": "0.1"}
            }
        ],
        "trialTemplate": {
            "primaryContainerName": "training-container",
            "trialParameters": [
                {"name": "n_estimators", "description": "", "reference": "n_estimators"},
                {"name": "max_depth",    "description": "", "reference": "max_depth"}
            ],
            "trialSpec": {
                "apiVersion": "batch/v1",
                "kind": "Job",
                "spec": {
                    "template": {
                        "spec": {
                            "containers": [{
                                "name": "training-container",
                                "image": "my-registry/train:latest",
                                "command": [
                                    "python", "train.py",
                                    "--n_estimators=${trialParameters.n_estimators}",
                                    "--max_depth=${trialParameters.max_depth}"
                                ]
                            }]
                        }
                    }
                }
            }
        }
    }
}

print("Katib Experiment config (apply with kubectl):")
print(json.dumps(katib_experiment, indent=2)[:1500] + "\n...")
print()
print("Deploy with: kubectl apply -f katib_experiment.yaml")
print("Monitor:     kubectl get experiment random-forest-hpo -n ml-team")

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Importing at top of component | Import error in container | All imports INSIDE the component function |
| Using local files in components | FileNotFoundError | Use Input/Output artifacts; no access to local disk |
| Forgetting packages_to_install | ModuleNotFoundError | List all needed packages in `@component(packages_to_install=[...])` |
| Using base_image without needed tools | Build fails | Use an image that already has what you need |
| Large data through parameters | Pipeline slow | Use Dataset artifacts, not parameter strings |
| Component timeout | Kubernetes kills pod | Set appropriate timeout in pipeline config |
| Not requesting GPU | Training on CPU | Use `set_accelerator_type` in pipeline definition |

## Interview Questions and Answers

In [ ]:
qa = [
    {"q": "Kubeflow Pipelines vs Apache Airflow — when to use each?",
     "a": """Kubeflow Pipelines:
- Each step runs in its own Docker container on Kubernetes
- Designed for ML workflows with GPU support, distributed training
- Artifacts (datasets, models) tracked automatically
- Katib for HPO, KServe for serving — full ML platform
- Requires Kubernetes cluster (more complex to set up)
- Best for: large-scale ML on K8s, GPU workloads, containerized steps

Apache Airflow:
- Tasks are Python functions, run on Airflow workers
- General-purpose: ETL, data pipelines, any scheduled work
- Better monitoring and alerting out of the box
- Simpler to get started (no Kubernetes needed)
- Best for: data pipelines, ETL, any scheduled workflows

Choose KFP: ML-first platform, team already on Kubernetes, need GPU scheduling
Choose Airflow: general workflow orchestration, non-ML steps alongside ML
Many teams use both: Airflow triggers KFP pipelines"""},

    {"q": "What is a KFP component and how does it differ from an Airflow task?",
     "a": """KFP Component:
- Runs inside its own Docker container on Kubernetes
- Isolated: has its own filesystem, memory, CPU/GPU allocation
- Communicates via artifacts (files) and parameters (scalars)
- Can be GPU-enabled, scaled independently
- Reproducible: same container image = same environment always

Airflow Task:
- Runs as a Python function on an Airflow worker machine
- Shares the worker's Python environment and filesystem
- Communicates via XComs (small data) or external storage
- Less isolation, but simpler to develop and debug
- Operators for many services (SQL, S3, Spark) built-in

Key difference: KFP = containerized, K8s-native, ML-artifact-aware
Airflow = Python-function-based, worker-pool model, general-purpose"""},

    {"q": "How does Katib work for hyperparameter optimization?",
     "a": """Katib is Kubeflow's HPO system. It works like this:

1. You define an Experiment (Kubernetes CRD) specifying:
   - Objective metric to optimize (roc_auc, maximize)
   - Search space (n_estimators: 50-500, max_depth: 3-20)
   - Search algorithm (Bayesian, Random, Grid, CMA-ES, etc.)
   - Trial template (what container to run for each trial)

2. Katib controller creates Trial objects (one per hyperparameter set)
3. Each Trial launches a Kubernetes Job (your training container)
4. Your training code prints metrics to stdout in Katib format:
   print('{"metric": [{"name": "roc_auc", "value": "0.95"}]}')
5. Katib's metrics collector reads stdout, updates the Experiment
6. Bayesian optimizer picks next Trial based on observed results

parallelTrialCount: run N trials simultaneously (uses more K8s resources)
maxTrialCount: budget (total number of trials)
goal: early stopping if metric reached

Katib vs Optuna: Katib = K8s-native, truly distributed; Optuna = simpler, in-process"""},
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 65)
    print()

## Summary

| Component | What It Does |
|-----------|-------------|
| KFP SDK (`kfp`) | Python library to define components + pipelines |
| `@component` | Decorator to turn a Python function into a containerized step |
| `@pipeline` | Decorator to define a pipeline connecting components |
| `Input[Dataset]` / `Output[Model]` | Typed artifact inputs/outputs |
| `kfp.compiler.Compiler()` | Compiles pipeline to YAML for deployment |
| `kfp.Client` | Submits pipelines to running Kubeflow cluster |
| Katib | Automated hyperparameter tuning on Kubernetes |
| KServe | Serving deployed models with autoscaling |

### Next Steps
1. **KFP quickstart**: [https://www.kubeflow.org/docs/components/pipelines/v2/tutorials/](https://www.kubeflow.org/docs/components/pipelines/v2/tutorials/)
2. **Local Kubeflow with kind**: [https://www.kubeflow.org/docs/started/installing-kubeflow/](https://www.kubeflow.org/docs/started/installing-kubeflow/)
3. **Next**: Learn model serving frameworks (FastAPI, BentoML, Ray Serve)